# Lens C — Topic Trends

Is global health research aligned with actual disease burden? This notebook compares publication volume against WHO Global Burden of Disease (GBD) data, identifies COVID-era displacement effects, and measures topic "fashionability."

### Analyses
1. **Research intensity ratio** — publication share ÷ DALY share by topic and year
2. **Structural vs. temporary COVID displacement** — pre/post trend lines
3. **Topic fashionability index** — spike height vs. sustained volume ratio
4. **NCD transition** — infectious vs. NCD publication ratio vs. mortality ratio 2010–2024

### Prerequisites
- WHO GBD data loaded into `gbd_burden` table (from `data/gbd/`)
- Topic-to-GBD mapping populated in `topic_burden_map` table

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)

DB = 'data/global_health.duckdb'
con = duckdb.connect(DB, read_only=True)

## Load core data

In [ ]:
# Topic taxonomy for labels
topic_labels = pd.read_csv('data/taxonomy/topic_taxonomy.csv')
cat_names = topic_labels.drop_duplicates('category_letter')[['category_letter', 'category_name']]
cat_map = dict(zip(cat_names['category_letter'], cat_names['category_name']))

# Publication counts by topic and year
topic_year_counts = con.execute("""
    SELECT topic_category, publication_year, COUNT(*) AS n_papers
    FROM works
    WHERE topic_category IS NOT NULL
    GROUP BY topic_category, publication_year
    ORDER BY topic_category, publication_year
""").fetchdf()

topic_year_counts['topic_name'] = topic_year_counts['topic_category'].map(cat_map)

# Total papers per year for share calculation
yearly_totals = topic_year_counts.groupby('publication_year')['n_papers'].sum().reset_index()
yearly_totals.columns = ['publication_year', 'total_papers']

topic_year_counts = topic_year_counts.merge(yearly_totals, on='publication_year')
topic_year_counts['pub_share'] = topic_year_counts['n_papers'] / topic_year_counts['total_papers']

print(f'Years covered: {topic_year_counts["publication_year"].min()}–{topic_year_counts["publication_year"].max()}')
print(f'Topic categories: {topic_year_counts["topic_category"].nunique()}')

## Check GBD data availability

If GBD data hasn't been loaded yet, the research intensity ratio (C1) and NCD transition (C4) analyses will use publication data only. Load GBD data into the `gbd_burden` table and re-run to include burden comparisons.

In [ ]:
gbd_count = con.execute('SELECT COUNT(*) FROM gbd_burden').fetchone()[0]
map_count = con.execute('SELECT COUNT(*) FROM topic_burden_map').fetchone()[0]
HAS_GBD = gbd_count > 0 and map_count > 0

if HAS_GBD:
    print(f'GBD data loaded: {gbd_count:,} rows')
    print(f'Topic-burden mappings: {map_count}')
    
    # Load GBD burden data — DALYs as primary, Deaths for sensitivity
    burden_data = con.execute("""
        SELECT tbm.topic_category, tbm.topic_name, g.year,
               g.measure, SUM(g.val) AS total_burden
        FROM topic_burden_map tbm
        JOIN gbd_burden g ON tbm.gbd_cause = g.cause
        WHERE g.metric = 'Number'
          AND g.sex = 'Both'
          AND g.age_group = 'All ages'
          AND g.region = 'Global'
        GROUP BY tbm.topic_category, tbm.topic_name, g.year, g.measure
    """).fetchdf()
    
    # Separate DALYs and Deaths for dual analysis
    burden_dalys = burden_data[burden_data['measure'] == 'DALYs'].copy()
    burden_deaths = burden_data[burden_data['measure'] == 'Deaths'].copy()
    
    # Compute DALY shares (primary)
    yearly_dalys = burden_dalys.groupby('year')['total_burden'].sum().reset_index()
    yearly_dalys.columns = ['year', 'total_dalys_all']
    burden_dalys = burden_dalys.merge(yearly_dalys, on='year')
    burden_dalys['daly_share'] = burden_dalys['total_burden'] / burden_dalys['total_dalys_all']
    
    # Compute death shares (sensitivity)
    yearly_deaths = burden_deaths.groupby('year')['total_burden'].sum().reset_index()
    yearly_deaths.columns = ['year', 'total_deaths_all']
    burden_deaths = burden_deaths.merge(yearly_deaths, on='year')
    burden_deaths['death_share'] = burden_deaths['total_burden'] / burden_deaths['total_deaths_all']
    
    HAS_DEATHS = len(burden_deaths) > 0
    
    print(f'\n  DALYs: {len(burden_dalys):,} topic-year rows')
    if HAS_DEATHS:
        print(f'  Deaths: {len(burden_deaths):,} topic-year rows (sensitivity analysis)')
else:
    print('⚠️  GBD data not yet loaded. Analyses C1 and C4 will be limited.')
    print('   To enable: run pipeline/07_gbd_burden.py after placing IHME CSVs in data/gbd/')
    print('   Also populate topic_burden_map with topic_category → GBD cause mappings.')

---
## C1. Research Intensity Ratio

Publication share ÷ DALY share by topic. Values > 1 mean a topic is "over-researched" relative to its disease burden; < 1 means "under-researched."

If GBD data is not available, this section shows publication share trends only.

In [ ]:
if HAS_GBD:
    # Merge publication shares with DALY shares (primary analysis)
    intensity = topic_year_counts.merge(
        burden_dalys[['topic_category', 'year', 'daly_share']],
        left_on=['topic_category', 'publication_year'],
        right_on=['topic_category', 'year'],
        how='inner'
    )
    intensity['research_intensity'] = intensity['pub_share'] / intensity['daly_share']
    
    # Latest year snapshot
    latest_year = intensity['publication_year'].max()
    latest = intensity[intensity['publication_year'] == latest_year].copy()
    latest = latest.sort_values('research_intensity', ascending=True)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    colors = ['#d62728' if x < 1 else '#2ca02c' for x in latest['research_intensity']]
    ax.barh(latest['topic_name'], latest['research_intensity'], color=colors)
    ax.axvline(x=1, color='black', linestyle='-', linewidth=1.5, label='Parity (1.0)')
    ax.set_xlabel('Research Intensity Ratio (pub share ÷ DALY share)')
    ax.set_title(f'Research Intensity Ratio by Topic ({latest_year})\n'
                 f'Red = under-researched relative to burden, Green = over-researched')
    ax.legend()
    fig.tight_layout()
    plt.show()
    
    # Trend over time for select topics
    interesting_topics = latest.nsmallest(3, 'research_intensity')['topic_name'].tolist() + \
                         latest.nlargest(3, 'research_intensity')['topic_name'].tolist()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    for topic in interesting_topics:
        td = intensity[intensity['topic_name'] == topic]
        ax.plot(td['publication_year'], td['research_intensity'], 'o-', label=topic, markersize=4)
    ax.axhline(y=1, color='black', linestyle='-', linewidth=1, alpha=0.5)
    ax.set_xlabel('Year')
    ax.set_ylabel('Research Intensity Ratio')
    ax.set_title('Research Intensity Ratio Over Time — Most Over- and Under-Researched Topics')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    fig.tight_layout()
    plt.show()
    
    # --- Sensitivity: Deaths-based intensity ---
    if HAS_DEATHS:
        intensity_deaths = topic_year_counts.merge(
            burden_deaths[['topic_category', 'year', 'death_share']],
            left_on=['topic_category', 'publication_year'],
            right_on=['topic_category', 'year'],
            how='inner'
        )
        intensity_deaths['research_intensity_deaths'] = (
            intensity_deaths['pub_share'] / intensity_deaths['death_share']
        )
        
        # Compare DALYs vs Deaths intensity for latest year
        latest_deaths = intensity_deaths[
            intensity_deaths['publication_year'] == latest_year
        ][['topic_name', 'research_intensity_deaths']].copy()
        
        comparison = latest[['topic_name', 'research_intensity']].merge(
            latest_deaths, on='topic_name', how='inner'
        )
        comparison = comparison.sort_values('research_intensity', ascending=True)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        y_pos = range(len(comparison))
        ax.barh([y - 0.2 for y in y_pos], comparison['research_intensity'],
                height=0.4, label='DALYs-based', color='#1f77b4', alpha=0.8)
        ax.barh([y + 0.2 for y in y_pos], comparison['research_intensity_deaths'],
                height=0.4, label='Deaths-based', color='#ff7f0e', alpha=0.8)
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(comparison['topic_name'])
        ax.axvline(x=1, color='black', linestyle='-', linewidth=1.5)
        ax.set_xlabel('Research Intensity Ratio')
        ax.set_title(f'Sensitivity: DALYs vs Deaths-Based Research Intensity ({latest_year})\n'
                     f'Divergence reveals topics where disability burden ≠ mortality burden')
        ax.legend()
        fig.tight_layout()
        plt.show()
else:
    # Without GBD: show publication share trends
    fig, ax = plt.subplots(figsize=(14, 6))
    top_topics = topic_year_counts.groupby('topic_name')['n_papers'].sum().nlargest(8).index
    for topic in top_topics:
        td = topic_year_counts[topic_year_counts['topic_name'] == topic]
        ax.plot(td['publication_year'], td['pub_share'], 'o-', label=topic, markersize=4)
    ax.set_xlabel('Year')
    ax.set_ylabel('Publication share')
    ax.set_title('Topic Publication Shares Over Time (Top 8 Topics)')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    fig.tight_layout()
    plt.show()
    print('\n⚠️  Load GBD data to compute research intensity ratios.')

---
## C2. Structural vs. Temporary COVID Displacement

Did COVID-19 permanently reshape the global health research landscape, or did displaced topics bounce back? We compare pre-COVID trends (2010–2019) with post-COVID recovery (2022–2024).

In [ ]:
# Compute pre-COVID average share (2017-2019) and post-COVID (2022-2024)
pre_covid = topic_year_counts[
    topic_year_counts['publication_year'].between(2017, 2019)
].groupby('topic_name')['pub_share'].mean().reset_index()
pre_covid.columns = ['topic_name', 'pre_covid_share']

# COVID peak (2020-2021)
covid_peak = topic_year_counts[
    topic_year_counts['publication_year'].between(2020, 2021)
].groupby('topic_name')['pub_share'].mean().reset_index()
covid_peak.columns = ['topic_name', 'covid_share']

# Post-COVID (2022-2024)
post_covid = topic_year_counts[
    topic_year_counts['publication_year'] >= 2022
].groupby('topic_name')['pub_share'].mean().reset_index()
post_covid.columns = ['topic_name', 'post_covid_share']

displacement = pre_covid.merge(covid_peak, on='topic_name', how='outer')
displacement = displacement.merge(post_covid, on='topic_name', how='outer').fillna(0)

# Displacement = change during COVID; Recovery = how much came back
displacement['covid_displacement'] = displacement['covid_share'] - displacement['pre_covid_share']
displacement['recovery'] = displacement['post_covid_share'] - displacement['covid_share']
displacement['net_change'] = displacement['post_covid_share'] - displacement['pre_covid_share']
displacement = displacement.sort_values('net_change', ascending=True)

# Classify: structural shift vs. temporary displacement
displacement['type'] = displacement['net_change'].apply(
    lambda x: 'Structural gain' if x > 0.005 else ('Structural loss' if x < -0.005 else 'Temporary')
)

fig, ax = plt.subplots(figsize=(14, 8))
colors = {'Structural gain': '#2ca02c', 'Structural loss': '#d62728', 'Temporary': '#7f7f7f'}
bar_colors = [colors[t] for t in displacement['type']]
ax.barh(displacement['topic_name'], displacement['net_change'], color=bar_colors)
ax.axvline(x=0, color='black', linewidth=1)
ax.set_xlabel('Net change in publication share (post-COVID minus pre-COVID)')
ax.set_title('COVID Displacement: Structural vs. Temporary Changes in Topic Share')

# Add legend manually
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in colors.items()]
ax.legend(handles=legend_elements, loc='lower right')
fig.tight_layout()
plt.show()

displacement[['topic_name', 'pre_covid_share', 'covid_share', 'post_covid_share', 'net_change', 'type']]

---
## C3. Topic Fashionability Index

Some topics spike dramatically then fade, while others grow steadily. The fashionability index measures volatility: `max share / median share`. High values indicate "trendy" topics; low values indicate stable, sustained research areas.

In [ ]:
fashionability = (
    topic_year_counts
    .groupby('topic_name')
    .agg(
        max_share=('pub_share', 'max'),
        median_share=('pub_share', 'median'),
        mean_share=('pub_share', 'mean'),
        std_share=('pub_share', 'std'),
        total_papers=('n_papers', 'sum'),
    )
)

# Fashionability = peak / median (higher = more volatile)
fashionability['fashionability'] = fashionability['max_share'] / fashionability['median_share']
# Coefficient of variation as alternative measure
fashionability['cv'] = fashionability['std_share'] / fashionability['mean_share']
fashionability = fashionability.sort_values('fashionability', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#d62728' if x > 2 else '#ff7f0e' if x > 1.5 else '#2ca02c'
          for x in fashionability['fashionability']]
fashionability['fashionability'].plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Fashionability Index (peak share ÷ median share)')
ax.set_title('Topic Fashionability Index\nHigh = trendy/volatile, Low = stable/sustained')
ax.axvline(x=1.5, color='gray', linestyle='--', alpha=0.5, label='Threshold: 1.5')
ax.legend()
fig.tight_layout()
plt.show()

fashionability.sort_values('fashionability', ascending=False)

---
## C4. NCD Transition

As low- and middle-income countries undergo epidemiological transitions, NCDs are rising while infectious diseases decline. Is the research landscape keeping pace?

We compare the ratio of infectious disease publications (C, D, E) to NCD publications (F) against mortality ratios from GBD data (if available).

In [ ]:
# Infectious disease topics: C (Infectious), D (HIV/TB/Malaria), E (NTDs)
# NCD topics: F (NCDs)
infectious_cats = ['C', 'D', 'E']
ncd_cats = ['F']

transition_data = topic_year_counts[
    topic_year_counts['topic_category'].isin(infectious_cats + ncd_cats)
].copy()

transition_data['disease_group'] = transition_data['topic_category'].apply(
    lambda x: 'Infectious' if x in infectious_cats else 'NCD'
)

transition_summary = (
    transition_data
    .groupby(['publication_year', 'disease_group'])['n_papers']
    .sum()
    .unstack(fill_value=0)
)

transition_summary['ratio'] = transition_summary.get('Infectious', 0) / transition_summary.get('NCD', 1).replace(0, 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Stacked area chart of paper counts
transition_summary[['Infectious', 'NCD']].plot.area(ax=ax1, alpha=0.7,
    color=['#d62728', '#2ca02c'])
ax1.set_xlabel('Year')
ax1.set_ylabel('Number of papers')
ax1.set_title('Infectious Disease vs. NCD Publications')
ax1.legend(loc='upper left')

# Ratio over time
ax2.plot(transition_summary.index, transition_summary['ratio'], 'o-', color='#1f77b4', linewidth=2, label='Publication ratio')

if HAS_GBD:
    # Overlay GBD mortality ratios (Deaths-based)
    gbd_infectious = burden_deaths[
        burden_deaths['topic_category'].isin(infectious_cats)
    ].groupby('year')['total_burden'].sum().reset_index()
    gbd_infectious.columns = ['year', 'infectious_deaths']

    gbd_ncd = burden_deaths[
        burden_deaths['topic_category'].isin(ncd_cats)
    ].groupby('year')['total_burden'].sum().reset_index()
    gbd_ncd.columns = ['year', 'ncd_deaths']

    gbd_ratio = gbd_infectious.merge(gbd_ncd, on='year', how='inner')
    gbd_ratio['death_ratio'] = gbd_ratio['infectious_deaths'] / gbd_ratio['ncd_deaths'].replace(0, 1)

    ax2.plot(gbd_ratio['year'], gbd_ratio['death_ratio'], 's--', color='#d62728',
             linewidth=2, label='Mortality ratio (GBD)', markersize=5)

    # Also show DALYs ratio
    gbd_infectious_d = burden_dalys[
        burden_dalys['topic_category'].isin(infectious_cats)
    ].groupby('year')['total_burden'].sum().reset_index()
    gbd_infectious_d.columns = ['year', 'infectious_dalys']

    gbd_ncd_d = burden_dalys[
        burden_dalys['topic_category'].isin(ncd_cats)
    ].groupby('year')['total_burden'].sum().reset_index()
    gbd_ncd_d.columns = ['year', 'ncd_dalys']

    gbd_ratio_d = gbd_infectious_d.merge(gbd_ncd_d, on='year', how='inner')
    gbd_ratio_d['daly_ratio'] = gbd_ratio_d['infectious_dalys'] / gbd_ratio_d['ncd_dalys'].replace(0, 1)

    ax2.plot(gbd_ratio_d['year'], gbd_ratio_d['daly_ratio'], '^--', color='#ff7f0e',
             linewidth=2, label='DALY ratio (GBD)', markersize=5)

ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='Parity')
ax2.set_xlabel('Year')
ax2.set_ylabel('Infectious / NCD ratio')
ax2.set_title('Infectious-to-NCD Ratio Over Time')
ax2.legend(fontsize=9)

fig.suptitle('The NCD Transition in Global Health Research', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

if not HAS_GBD:
    print('\n⚠️  Load GBD data to overlay mortality and DALY ratios on this chart.')

---
## Save Results to DuckDB

In [ ]:
con.close()

con_write = duckdb.connect(DB)

# Topic-year publication counts
con_write.execute('CREATE OR REPLACE TABLE topic_year_counts AS SELECT * FROM ?', [topic_year_counts])

# Research intensity ratios (if GBD data available)
if HAS_GBD:
    con_write.execute('CREATE OR REPLACE TABLE burden_research_ratio AS SELECT * FROM ?', [intensity])
    if HAS_DEATHS:
        con_write.execute('CREATE OR REPLACE TABLE burden_research_ratio_deaths AS SELECT * FROM ?', [intensity_deaths])

# COVID displacement analysis
con_write.execute('CREATE OR REPLACE TABLE covid_displacement AS SELECT * FROM ?', [displacement])

con_write.close()
print('Lens C results saved to DuckDB.')